# Aula 3 · A cor do rio e a maré

**8 minutos (12:35–12:43) · aula guiada, em pares** · a seguir: a Prática 3

**O que vamos fazer:** usar o que aprenderam num assunto novo: a **qualidade da água**. Vamos tentar prever se a água do rio está **turva** (barrenta) de manhã, a partir de dados. Foi isto que fizemos no projecto EYES TO EARTH.

**Os dados:** `cor_do_rio_e_mare.csv` (2 205 linhas: uma manhã num troço do rio, de 2017 a 2026).  
**Treino e teste:** **treino** = as manhãs de 2017 a 2022 (1 214); **teste** = as de 2023 a 2026 (991).  
**O algoritmo:** floresta aleatória com 100 árvores (classificação: água turva ou não).

| Palavra nova | O que quer dizer |
| --- | --- |
| **turbidez** | a água barrenta, cheia de terra fina (sedimentos) |
| **NDTI** | o índice de turbidez que o satélite mede (Módulo 4): mais alto = água mais barrenta |
| **troço** | um pedaço do rio; aqui há três: o troço médio, a jusante e a foz (onde o rio chega ao mar) |
| **turva** | o rótulo: 1 = mais barrenta do que o normal nesse troço; 0 = mais limpa |
| **maré** | o mar sobe e desce duas vezes por dia; mede-se em metros (0 = o nível médio do mar) |

**Como trabalhar:** esta é a **aula guiada**: o facilitador mostra no projector e vocês acompanham.
Cliquem numa célula de código (as cinzentas) e carreguem em **Shift + Enter** para a correr, sempre **por ordem**, de
cima para baixo. A seguir vem a **prática**, em que são vocês a escrever o código. Guardem com **Ctrl + S**.

## Passo 1 · Abrir as ferramentas

In [ ]:
from ferramentas import *      # as ferramentas da aula: corram sem mudar nada
from sklearn.ensemble import RandomForestClassifier
print("Ferramentas prontas.")

## Passo 2 · Dez anos de manhãs no rio

Cada linha é uma manhã em que o satélite Sentinel-2 viu o rio sem nuvens, num dos três troços, de 2017 a 2026. `ndti` é a turbidez medida; `caudal_m3s` e `chuva_7d_mm` são o caudal e a chuva dos últimos 7 dias; `altura_da_mare_m` é a maré à hora do satélite. A coluna `turva` é o **rótulo**.

In [ ]:
rio = pd.read_csv(encontrar("cor_do_rio_e_mare.csv"))
print("Linhas na tabela:", numero(len(rio)), "(manhãs de 2017 a 2026, em 3 troços)")
print("Manhãs com a água turva:", percentagem(rio["turva"].mean()))
rio.head()

## Passo 3 · Separar no tempo: o exame são os anos mais recentes

Queremos um modelo para o futuro. Por isso treinamos com **2017 a 2022** e fazemos o exame com **2023 a 2026**, anos que o modelo nunca vê no treino.

In [ ]:
treino = rio[rio["ano"] <= 2022]
teste = rio[rio["ano"] >= 2023]
print("Manhãs de treino (2017 a 2022):", numero(len(treino)))
print("Manhãs de teste (2023 a 2026): ", numero(len(teste)))

## Passo 4 · Primeira tentativa: o caudal, a chuva e o mês

Parece lógico: depois de muita chuva, o rio traz mais terra. Damos à floresta aleatória (Tema 2) três pistas: o caudal, a chuva dos últimos 7 dias e o mês.

In [ ]:
PISTAS_SEM_MARE = ["caudal_m3s", "chuva_7d_mm", "mes"]
floresta = RandomForestClassifier(n_estimators=100, random_state=0)
floresta.fit(treino[PISTAS_SEM_MARE], treino["turva"])
nota_sem_mare = acerto(teste["turva"], floresta.predict(teste[PISTAS_SEM_MARE]))
sempre_turva = teste["turva"].mean()        # a nota de quem diz sempre «turva», sem modelo nenhum
print("Acerto no teste, sem a maré:   ", percentagem(nota_sem_mare))
print("Se disséssemos sempre «turva»: ", percentagem(sempre_turva))

A floresta acerta **menos** do que dizer sempre «turva». Com estas pistas, a máquina **não aprendeu nada útil**, e mais árvores não resolvem. **Falta uma pista.**

## Passo 5 · A pista que faltava: a maré

A foz do Incomáti fica no mar, e o mar sobe e desce duas vezes por dia. A tabela tem a altura da maré à hora em que o satélite passou (cerca das 9:30). Vejamos a turbidez em cada altura da maré.

In [ ]:
ver_mare(rio)

Com a **maré baixa**, a água está quase sempre turva; com a **maré alta**, quase sempre mais limpa. Uma explicação simples: na maré alta entra água do mar, mais limpa; na maré baixa sai a água barrenta do rio.

No projecto EYES TO EARTH aconteceu o mesmo: o modelo da água barrenta só começou a funcionar quando lhe demos a maré. **Na Prática 3, são vocês que juntam a maré ao modelo.**

## O que aprendemos

- Treinámos uma **classificação** (turva ou não) com os dados de um assunto novo: a qualidade da água.
- Com o caudal, a chuva e o mês, o modelo acerta **menos** do que «dizer sempre turva». Comparem sempre o modelo com «não fazer nada».
- O gráfico mostrou a pista que faltava: a **maré**. Escolher as pistas certas vale mais do que o algoritmo.

## Se aparecer um erro

Leiam a **última linha** da mensagem vermelha.

| A última linha diz | O que fazer |
| --- | --- |
| `NameError: name '...' is not defined` | Saltaram uma célula. Corram as células por ordem, a partir do Passo 1. |
| `FileNotFoundError: Não encontrei ...` | Abram o notebook a partir da pasta `Workshop_2/2_pratica`. |
| `ModuleNotFoundError` | Falta uma biblioteca neste computador: chamem um facilitador. |
| `NotFittedError` ou «O modelo ainda não foi treinado» | Falta treinar o modelo: corram a célula com `.fit(...)`. |
| `SyntaxError` | Falta uma aspa, um parêntese ou uma vírgula. Comparem com o exemplo. |
| «Falta preencher» | Troquem os `...` pela vossa resposta. |

Se nada resultar: menu **Kernel**, depois **Restart Kernel and Run All Cells**.